In [42]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adhoppin/blood-cell-detection-datatset")

print("Path to dataset files:", path)

Path to dataset files: /home/codespace/.cache/kagglehub/datasets/adhoppin/blood-cell-detection-datatset/versions/1428


In [43]:
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import os
import torch
from PIL import Image
transforms = transforms.Compose([ 
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],std =[0.5, 0.5, 0.5])
])

In [44]:
class BloodCellDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        images_dir = os.path.join(root_dir, "train", "images")
        labels_dir = os.path.join(root_dir, "train", "labels")

        self.image_paths = sorted(
            os.path.join(images_dir, fname)
            for fname in os.listdir(images_dir)
            if fname.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        self.labels = sorted(
            os.path.join(labels_dir, fname)
            for fname in os.listdir(labels_dir)
            if fname.lower().endswith((".txt", ".csv"))
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        # label is txt file path, you can read it and process as needed
        with open(self.labels[idx], "r") as f:
            lines = f.readlines()
    
        class_id = int(lines[0].strip().split()[0])  # Assuming the first line contains the class ID

        label = class_id
        if self.transform:
            image = self.transform(image)
        return image, label

    def load_image(self, img_path):
        return Image.open(img_path).convert("RGB")   

In [45]:
dataset = BloodCellDataset(root_dir=path, transform=transforms)
sample_Image, sample_Label = dataset.__getitem__(44)
print("Dataset_sample:", sample_Image, sample_Label)



Dataset_sample: tensor([[[-0.4980, -0.0980, -0.1137,  ..., -0.1373, -0.1373, -0.1373],
         [-0.0824,  0.6235,  0.6314,  ...,  0.6392,  0.6392,  0.6392],
         [-0.1059,  0.6235,  0.6157,  ...,  0.6471,  0.6471,  0.6471],
         ...,
         [-0.1373,  0.6706,  0.6471,  ...,  0.5137,  0.5529,  0.5922],
         [-0.1373,  0.6627,  0.6471,  ...,  0.5451,  0.5843,  0.6157],
         [-0.1373,  0.6627,  0.6392,  ...,  0.5686,  0.6078,  0.6235]],

        [[-0.5608, -0.1529, -0.1529,  ..., -0.0980, -0.0980, -0.0980],
         [-0.1451,  0.5686,  0.5922,  ...,  0.6863,  0.6863,  0.6863],
         [-0.1608,  0.5686,  0.5765,  ...,  0.6941,  0.6941,  0.6941],
         ...,
         [-0.1294,  0.6784,  0.6549,  ...,  0.4510,  0.5137,  0.5765],
         [-0.1373,  0.6706,  0.6471,  ...,  0.4824,  0.5451,  0.6000],
         [-0.1373,  0.6706,  0.6471,  ...,  0.5059,  0.5686,  0.6078]],

        [[-0.5765, -0.1843, -0.1922,  ..., -0.0980, -0.0980, -0.0980],
         [-0.1608,  0.5294,  

In [46]:
from torch.utils.data import random_split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")



Train size: 535, Val size: 114, Test size: 116


In [47]:
# dataloader on train and test set
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


Train batches: 17, Val batches: 4, Test batches: 4


In [48]:
# simple CNN model
import torch.nn as nn
import torch.nn.functional as F
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 56 * 56, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 56 * 56)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
model = SimpleCNN(num_classes=3)
print(model)


SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=200704, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=3, bias=True)
)


In [ ]:
# Train the model
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
# Evaluate the model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")



Epoch 1/10, Loss: 2.2116
Epoch 2/10, Loss: 0.5961
Epoch 3/10, Loss: 0.5159
Epoch 4/10, Loss: 0.4430
Epoch 5/10, Loss: 0.3440
Epoch 6/10, Loss: 0.3044
Epoch 7/10, Loss: 0.2462
Epoch 8/10, Loss: 0.1968
Epoch 9/10, Loss: 0.1150
Epoch 10/10, Loss: 0.0814
Test Accuracy: 0.7586


: 